# 02 MVP 150 Eval Colab

本 notebook 用于评测本地生成并同步到 Google Drive 的 150 条 baseline MVP 音频：clean、noise、reverb、far_field、dropout 各 30 条。

音频应先在本地用 `scripts/create_mvp_eval_audio.py --force` 生成，再把 `data/mvp_eval/audio/` 和 `data/jsonl/baseline_mvp_150.local.jsonl` 同步到 Drive 项目目录。


In [ ]:
# 挂载 Google Drive。所有输入音频、manifest、prediction 和指标都放在 Drive 中。
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# 安装最小依赖。only-if-needed 可以降低 Colab 预装包被过度升级的风险。
# qwen-asr 是 Qwen3-ASR 官方推理包。
# 不要安装未固定版本的 pandas；Colab 当前依赖 pandas==2.2.2，pandas 3.x 会和 google-colab/db-dtypes/gradio 冲突。
%pip -q install --upgrade --upgrade-strategy only-if-needed qwen-asr soundfile huggingface_hub

# 如果之前误升级到 pandas 3.x，这行会把它修回 Colab 兼容版本。
%pip -q install pandas==2.2.2



In [ ]:
# 项目路径和评测参数。
# 如果你的 Drive 目录名不同，只需要改 PROJECT_DIR。
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3-asr')
AUDIO_ROOT = PROJECT_DIR

# 默认使用本地生成脚本产出的 manifest。也兼容你手动改名后的 baseline_mvp_150.jsonl。
MANIFEST_CANDIDATES = [
    PROJECT_DIR / 'data/jsonl/baseline_mvp_150.local.jsonl',
    PROJECT_DIR / 'data/jsonl/baseline_mvp_150.jsonl',
]
MANIFEST = next((path for path in MANIFEST_CANDIDATES if path.exists()), MANIFEST_CANDIDATES[0])

OUTPUT_DIR = PROJECT_DIR / 'outputs/baseline_mvp_150'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 归一化后的 Colab manifest。推理脚本会读取这个文件，避免本地绝对路径带到 Colab 后失效。
COLAB_MANIFEST = OUTPUT_DIR / 'baseline_mvp_150.colab.jsonl'
PREDICTIONS = OUTPUT_DIR / 'predictions.qwen3_asr_base.mvp_150.jsonl'
SCORED = OUTPUT_DIR / 'predictions.qwen3_asr_base.mvp_150.scored.jsonl'
METRICS = OUTPUT_DIR / 'metrics.qwen3_asr_base.mvp_150.json'
SCENARIO_CSV = OUTPUT_DIR / 'metrics_by_scenario.qwen3_asr_base.mvp_150.csv'

MODEL_ID = 'Qwen/Qwen3-ASR-1.7B'
DTYPE = 'float16'
DEVICE_MAP = 'cuda:0'
MAX_NEW_TOKENS = 256
MAX_INFERENCE_BATCH_SIZE = 1
LANGUAGE = 'English'

# 0 表示跑完整 150 条。调试时可以先改成 5 或 10。
LIMIT = 0

print('PROJECT_DIR =', PROJECT_DIR)
print('MANIFEST =', MANIFEST)
print('manifest exists =', MANIFEST.exists())
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
# 预览本地生成脚本写出的 stats。
# 这里重点确认 profile 是否为 hard，以及各 degraded 场景是否已经明显降低质量。
import json
STATS_CANDIDATES = [
    PROJECT_DIR / 'data/jsonl/baseline_mvp_150_stats.local.json',
    PROJECT_DIR / 'data/jsonl/baseline_mvp_150_stats.json',
]
STATS = next((path for path in STATS_CANDIDATES if path.exists()), None)

if STATS is None:
    print('未找到 stats 文件；可以继续跑 manifest 校验，但建议同步 baseline_mvp_150_stats.local.json。')
else:
    stats = json.loads(STATS.read_text(encoding='utf-8'))
    print('stats =', STATS)
    print('profile =', stats.get('profile'))
    print('scenario_counts =', stats.get('scenario_counts'))
    print('degradation_stats =')
    print(json.dumps(stats.get('degradation_stats', {}), ensure_ascii=False, indent=2))


In [ ]:
# 读取、校验并归一化 manifest。
# 这里会检查 5 个场景是否各 30 条，并确认每个音频文件在 Drive 中存在。
import json
from collections import Counter

EXPECTED_COUNTS = {
    'clean': 30,
    'noise': 30,
    'reverb': 30,
    'far_field': 30,
    'dropout': 30,
}

def resolve_audio_for_colab(audio_value: str) -> Path:
    """把 manifest 中的本地路径解析成 Colab/Drive 中真实存在的路径。"""
    path = Path(audio_value)
    if path.exists():
        return path

    # 相对路径：优先按项目根目录解析。
    if not path.is_absolute():
        for candidate in [PROJECT_DIR / path, MANIFEST.parent / path]:
            if candidate.exists():
                return candidate

    # 本地 Mac 绝对路径：截取 data/... 后映射到 Drive 项目根目录。
    parts = path.parts
    if 'data' in parts:
        data_index = parts.index('data')
        candidate = PROJECT_DIR / Path(*parts[data_index:])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f'找不到音频文件: {audio_value}')

rows = []
with MANIFEST.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

assert len(rows) == 150, f'期望 150 条，实际 {len(rows)} 条'
counts = Counter(row.get('scenario') for row in rows)
assert dict(counts) == EXPECTED_COUNTS, f'场景计数不匹配: {counts}'

normalized_rows = []
for row in rows:
    for field in ['audio', 'answer', 'language', 'scenario']:
        assert field in row, f'缺少字段 {field}: {row}'
    out = dict(row)
    out['audio'] = str(resolve_audio_for_colab(str(row['audio'])))
    normalized_rows.append(out)

with COLAB_MANIFEST.open('w', encoding='utf-8') as f:
    for row in normalized_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('场景计数:', dict(counts))
print('已写入 Colab manifest:', COLAB_MANIFEST)
print('第一条样本:', normalized_rows[0])


In [ ]:
# 登录 Hugging Face。如果下载限流或需要认证，可以在这里登录后重试。
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
# 运行 Qwen3-ASR baseline 推理。
# 使用 subprocess 列表参数，避免 shell 变量展开问题。
import subprocess
import sys

cmd = [
    sys.executable,
    'inference/qwen3_asr_base_infer.py',
    '--manifest', str(COLAB_MANIFEST),
    '--audio-root', str(AUDIO_ROOT),
    '--output-jsonl', str(PREDICTIONS),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--max-inference-batch-size', str(MAX_INFERENCE_BATCH_SIZE),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--language', LANGUAGE,
]
if LIMIT > 0:
    cmd.extend(['--limit', str(LIMIT)])

print('运行命令:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)
print('prediction JSONL =', PREDICTIONS)



In [ ]:
# 快速检查推理输出：统计错误数和空输出数。
prediction_rows = []
with PREDICTIONS.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            prediction_rows.append(json.loads(line))

error_count = sum(1 for row in prediction_rows if row.get('error'))
empty_count = sum(1 for row in prediction_rows if not str(row.get('prediction') or '').strip())
print('prediction rows =', len(prediction_rows))
print('error_count =', error_count)
print('empty_count =', empty_count)
print('前 3 条输出:')
for row in prediction_rows[:3]:
    print({
        'scenario': row.get('scenario'),
        'answer': row.get('answer'),
        'prediction': row.get('prediction'),
        'error': row.get('error'),
    })


In [ ]:
# 计算 WER/CER，并输出整体指标和按场景指标。
eval_cmd = [
    sys.executable,
    'evaluation/eval_wer.py',
    '--predictions-jsonl', str(PREDICTIONS),
    '--scored-jsonl', str(SCORED),
    '--metrics-json', str(METRICS),
    '--metrics-by-scenario-csv', str(SCENARIO_CSV),
]

print('运行命令:')
print(' '.join(eval_cmd))
subprocess.run(eval_cmd, cwd=str(PROJECT_DIR), check=True)

print('metrics JSON =', METRICS)
print(METRICS.read_text(encoding='utf-8'))


In [ ]:
# 展示 scenario-level CSV。这个表就是本轮 MVP 150 baseline 的主要结果入口。
import pandas as pd

pd.read_csv(SCENARIO_CSV)
